# Association utility at fixed false-merge operating points

This notebook reproduces the nested task-held-out experiment in `scripts/run_association_operating_points.py`. The same held-out candidate gallery is used for command cosine, behavior, structure-aware, mode-aware, and length-control rankers. Thresholds are selected only from nested training-task predictions to target 10%, 20%, or 30% macro false merges.

The positive label is a shared benchmark-qualified CTF task, a same-challenge proxy. It does not mean a shared intrusion or operator. Human terminal and Agent tool-call collection remain confounded.

In [ ]:
from pathlib import Path
import json, subprocess, sys
ROOT = Path.cwd()
if (ROOT / 'research' / 'scripts').is_dir(): ROOT = ROOT / 'research'
if not (ROOT / 'scripts' / 'run_association_operating_points.py').exists(): ROOT = ROOT.parent
assert (ROOT / 'scripts' / 'run_association_operating_points.py').exists()
PYTHON = ROOT.parent / '.venv' / 'bin' / 'python'
print('Repository:', ROOT)
print('Python:', subprocess.check_output([str(PYTHON), '--version'], text=True).strip())
plot = subprocess.run([str(PYTHON), 'scripts/plot_association_operating_points.py'], cwd=ROOT, check=True, text=True, capture_output=True)
print(plot.stdout[-2000:])


## Recompute nested folds

The calculation is CPU-only. Each outer test fold is evaluated once. No execution-mode filtering is applied to candidate generation.

In [ ]:
run = subprocess.run([str(PYTHON), 'scripts/run_association_operating_points.py', '--bootstraps', '5000'], cwd=ROOT, check=True, text=True, capture_output=True)
print(run.stdout[-6000:])

In [ ]:
import pandas as pd
from IPython.display import display, Image
result = json.loads((ROOT / 'results' / 'association_operating_points_v1.json').read_text())
summary = result['false_merge_targets']['20']['summary']
rows = []
for method, metrics in summary.items():
    rows.append({'method': method, 'false split': metrics['false_split_rate']['estimate'],
                 'false split 95% CI': metrics['false_split_rate']['ci_95'],
                 'false merge': metrics['false_merge_rate']['estimate'],
                 'false merge 95% CI': metrics['false_merge_rate']['ci_95'],
                 'balanced accuracy': metrics['balanced_accuracy']['estimate']})
display(pd.DataFrame(rows).set_index('method').round(3))
print(result['interpretation_guard'])

In [ ]:
display(pd.DataFrame([{'method': m, **{k: result['ranking_summary'][m][k]['estimate'] for k in ('hit_at_1','mrr','recall_at_5','ndcg_at_5')}} for m in result['ranking_summary']]).set_index('method').round(3))
display(Image(filename=str(ROOT / 'results' / 'figures' / 'association_operating_points_v1.png')))

## Reading the result

Compare methods at the same training false-merge ceiling, then inspect the realized test false-merge rate. A lower false-split rate is useful only with its accompanying false-merge rate and retrieval metrics. Every association claim in this dataset remains a CTF task-proxy result.